# Prediksi Risiko Gagal Bayar Kredit Konsumen (DSC LOGIKA UI 2026)

**Tujuan.** Memprediksi probabilitas sebuah pinjaman mencapai DPD >= 90 dalam 12 bulan setelah origination. Metrik kompetisi: **KS statistic**.

**Pendekatan singkat**
1. Gabungkan 8 tabel sumber menjadi satu baris fitur per `application_id`, **hanya dari informasi sebelum `decision_datetime`** (aturan leakage ditegakkan oleh satu fungsi `join_before` yang dipakai semua modul).
2. Tiga lapis fitur: **v0** (ringkasan tiap tabel), **v1** (e-commerce, bureau, repayment, previous yang lebih rinci + rasio lintas tabel), dan **ec3** (e-commerce v2: jendela pendek, pola waktu, tren, pembatalan, indeks harga, ritme belanja).
3. Model **LightGBM**, validasi **5-fold stratified**, prediksi akhir = rata-rata **5 seed** (variansi antar seed besar pada data berisik ini).
4. Model pendamping **regresi logistik** (Pipeline scikit-learn) dan **blend peringkat** 0.7 LightGBM : 0.3 regresi logistik. Dua file submission dihasilkan: LightGBM saja dan blend.
5. **Bagian 13-19 (tahap peningkatan akhir)**: 10 seed LightGBM, pemangkasan fitur bersarang, uji kurva belajar dan *refit* seluruh data, lalu perbandingan resep blend terhadap blend saat ini dengan bootstrap berpasangan.

**Kepatuhan aturan.** Hanya data dari panitia (tanpa data eksternal), semua fitur memakai cutoff `decision_datetime`, ambang statistik (normalisasi skor bureau, median harga per kategori) dihitung dari **train saja tanpa target**, dan notebook berjalan dari nol tanpa file perantara.

**Cara menjalankan.** Run All dari atas ke bawah. Langkah-langkah penting mencetak pesan konfirmasi ketika berhasil. Perkiraan total waktu: sekitar 70-100 menit.

## 1. Setup

### 1.1 Impor library
Memuat library yang dipakai (pandas/numpy untuk data, LightGBM untuk model, scipy/sklearn untuk metrik dan fold) dan mencetak versinya agar hasil dapat direproduksi.

In [ ]:
import os, glob, sys, time, warnings
import numpy as np
import pandas as pd
import scipy, sklearn
import lightgbm as lgb
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)

print('Library berhasil diimpor')
print('  python     ', sys.version.split()[0])
for m in (np, pd, scipy, sklearn, lgb):
    print(f'  {m.__name__:<11s}{m.__version__}')

### 1.2 Konfigurasi
Semua pengaturan penting ada di satu tempat: lokasi data (`None` = dicari otomatis), seed, jumlah fold, parameter LightGBM, kolom yang dibuang dari model, dan daftar seed untuk rata-rata akhir (`BAG_SEEDS`).

- `NOISE_PREFIX` / `NOISE_COLS`: awalan kolom dari grup fitur yang tidak berkontribusi pada ablasi (rekening, riwayat bayar 6 bulan, flag `has_*`) dan kolom konstan; dibuang dari model.
- Parameter dibuat `deterministic=True` supaya hasil sama di setiap run.

In [ ]:
DATA_DIR = os.environ.get('DATA_DIR')            # None = cari otomatis, atau isi path folder data
SUB_DIR = 'submissions'
SEED, N_FOLDS = 42, 5
BAG_SEEDS = (42, 43, 44, 45, 46)                 # prediksi akhir = rata-rata model dari seed-seed ini
BASE_DROP = ['customer_id', 'is_train', 'target', 'decision_datetime']
NOISE_PREFIX = ('acc_', 'acc3_', 'rep6_', 'has_')                  # grup tanpa kontribusi pada ablasi
NOISE_COLS = ['term_months', 'has_bur', 'has_ec']                  # kolom konstan

LGB_PARAMS = dict(
    objective='binary', n_estimators=5000, learning_rate=0.03,
    num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.7, reg_lambda=5.0, n_jobs=-1, verbose=-1,
    deterministic=True, force_row_wise=True,
)
EARLY_STOP = 200

np.random.seed(SEED)
os.makedirs(SUB_DIR, exist_ok=True)
print('Konfigurasi siap')
print(f'  folds={N_FOLDS} | seed bagging={BAG_SEEDS} | learning_rate={LGB_PARAMS["learning_rate"]} | early stopping={EARLY_STOP}')

## 2. Data

### 2.1 Muat 8 tabel dan sample submission
`DT_COLS` mendaftarkan kolom tanggal tiap tabel agar langsung di-parse saat dibaca. Folder data dicari otomatis (termasuk `/kaggle/input`) memakai `glob`, lalu semua tabel dan `sample_submission.csv` dibaca dalam satu langkah.

In [ ]:
# [Data] Daftar kolom datetime per tabel
APP_DT = ['application_datetime', 'decision_datetime']
DT_COLS = {
    'train_applications': APP_DT,
    'test_applications': APP_DT,
    'customer_profiles': [],
    'previous_applications': APP_DT,
    'repayment_history': ['due_date', 'last_payment_datetime'],
    'account_behavior': ['month'],
    'bureau_observations': ['report_datetime', 'available_datetime'],
    'ecommerce_transactions': ['transaction_datetime'],
}

if DATA_DIR is None:
    hits = [h for root in ('.', 'data', '/kaggle/input', '/content')
            for h in glob.glob(f'{root}/**/train_applications.csv', recursive=True)]
    assert hits, 'train_applications.csv tidak ditemukan. Isi DATA_DIR di konfigurasi.'
    DATA_DIR = os.path.dirname(hits[0])

T = {n: pd.read_csv(os.path.join(DATA_DIR, f'{n}.csv'), parse_dates=c) for n, c in DT_COLS.items()}
SUB = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
tr, te, prof = T['train_applications'], T['test_applications'], T['customer_profiles']

print('DATA_DIR =', DATA_DIR)
for n, d in T.items():
    print(f'  {n:<24s}{d.shape[0]:>10,} baris x {d.shape[1]:>2} kolom')
print('8 tabel dan sample_submission berhasil dimuat')

### 2.2 Ringkasan kualitas tabel
Ukuran, persentase nilai kosong, dan jumlah baris duplikat tiap tabel, sebagai pemeriksaan awal kualitas data.

In [ ]:
# [Data] Ringkasan ukuran, missing, dan duplikasi tiap tabel
summary = pd.DataFrame({
    n: {'baris': len(d), 'kolom': d.shape[1],
        'missing_%': round(d.isna().mean().mean() * 100, 2),
        'baris_duplikat': int(d.duplicated().sum())}
    for n, d in T.items()}).T
display(summary)
print('Ringkasan kualitas data ditampilkan')

### 2.3 Cek integritas ID
Memastikan `application_id` unik, tidak ada ID train yang muncul di test, semua customer punya profil, dan ID test **persis sama** dengan `sample_submission`. Jika ada yang gagal, notebook berhenti di sini.

In [ ]:
# [Data] Cek integritas ID dan kecocokan train/test/profil
checks = {
    'application_id unik (train)': tr.application_id.is_unique,
    'application_id unik (test)': te.application_id.is_unique,
    'tidak ada ID train yang muncul di test': not (set(tr.application_id) & set(te.application_id)),
    'customer_id unik (profiles)': prof.customer_id.is_unique,
    'ID test == ID sample_submission': set(te.application_id) == set(SUB.application_id),
    'semua customer train punya profil': bool(tr.customer_id.isin(prof.customer_id).all()),
    'semua customer test punya profil': bool(te.customer_id.isin(prof.customer_id).all()),
}
res = pd.Series(checks, name='lolos').to_frame()
display(res)
assert res['lolos'].all(), 'Ada cek integritas yang gagal'
print('Semua cek integritas ID lolos')

### 2.4 Gambaran target dan waktu, lalu gabung train + test
Mencetak proporsi target, rentang waktu keputusan train vs test (rentang sama = test adalah split acak, bukan periode lebih baru), dan seberapa banyak customer test yang ada di train. Lalu train dan test digabung menjadi `ALL` agar fitur dibangun dengan aturan **yang sama persis** untuk keduanya.

In [ ]:
# [EDA] Target dan waktu: target rate, rentang keputusan, overlap customer
print(f'Target rate train : {tr.target.mean():.4%}  (n={len(tr):,}, positif={tr.target.sum():,})')
print('Train decision    :', tr.decision_datetime.min(), '->', tr.decision_datetime.max())
print('Test  decision    :', te.decision_datetime.min(), '->', te.decision_datetime.max())
print(f'Customer test yang juga ada di train: {te.customer_id.isin(tr.customer_id).mean():.2%}')
print('Aplikasi per customer (train):', tr.groupby('customer_id').size().agg(['mean', 'max']).round(2).to_dict())


ALL = pd.concat([tr.assign(is_train=1), te.assign(is_train=0)], ignore_index=True)
print(f'ALL: {len(ALL):,} aplikasi (train {len(tr):,} + test {len(te):,})')

## 3. Penegak aturan leakage: `join_before`
Satu fungsi yang dipakai **semua** modul fitur: menggabungkan tabel historis ke tiap aplikasi lalu hanya menyimpan baris yang terjadi **sebelum `decision_datetime`** (atau pada saat `available_datetime` untuk bureau). Dengan begitu tidak ada fitur yang bisa melihat masa depan.

In [ ]:
# [Fitur] Fungsi filter waktu: join lalu ambil baris sebelum cutoff
def join_before(A, tbl, tcol, key='customer_id', le=False):
    a = A[list(dict.fromkeys(['application_id', key, 'decision_datetime']))] \
        .rename(columns={'decision_datetime': 'cutoff'})
    t = tbl.drop(columns=[c for c in ['application_id', 'customer_id'] if c in tbl.columns and c != key])
    m = a.merge(t, on=key, how='inner')
    keep = (m[tcol] <= m['cutoff']) if le else (m[tcol] < m['cutoff'])
    return m[keep]

print('join_before siap (filter cutoff: transaksi harus < decision_datetime)')

## 4. Fitur v0: ringkasan tiap tabel
Satu fungsi per sumber data. Masing-masing menghasilkan satu baris per `application_id`.

### 4.1 Aplikasi + profil customer
Rasio kemampuan bayar (cicilan/penghasilan, pinjaman/penghasilan tahunan), beban bunga, penghasilan per tanggungan, dan waktu keputusan.

In [ ]:
# [Fitur] Modul aplikasi + profil
def f_base(A, prof):
    d = A.merge(prof, on='customer_id', how='left')
    inc = d.declared_monthly_income.replace(0, np.nan)
    d['inst_to_income'] = d.proposed_monthly_installment / inc
    d['loan_to_annual_income'] = d.loan_amount / (inc * 12)
    d['total_repay'] = d.proposed_monthly_installment * d.term_months
    d['interest_burden'] = d.total_repay / d.loan_amount - 1
    d['income_per_dependent'] = d.declared_monthly_income / (d.dependents + 1)
    d['tenure_years'] = d.employment_tenure_months / 12
    d['decision_lag_h'] = (d.decision_datetime - d.application_datetime).dt.total_seconds() / 3600
    d['dec_dow'] = d.decision_datetime.dt.dayofweek
    d['dec_hour'] = d.decision_datetime.dt.hour
    cols = ['application_id', 'loan_amount', 'term_months', 'annual_interest_rate',
            'proposed_monthly_installment', 'loan_purpose', 'age', 'region', 'employment_type',
            'employment_tenure_months', 'declared_monthly_income', 'dependents', 'housing_status',
            'inst_to_income', 'loan_to_annual_income', 'total_repay', 'interest_burden',
            'income_per_dependent', 'tenure_years', 'decision_lag_h', 'dec_dow', 'dec_hour']
    return d[cols].set_index('application_id')

print('f_base siap')

### 4.2 Aplikasi sebelumnya
Jumlah dan rasio pengajuan disetujui/ditolak/dibatalkan, rata-rata nominal, dan jarak waktu ke pengajuan terakhir.

In [ ]:
# [Fitur] Modul previous_applications
def f_previous(A, P):
    m = join_before(A, P, 'decision_datetime')
    m = m.assign(is_appr=(m.decision == 'approved').astype(int),
                 is_rej=(m.decision == 'rejected').astype(int),
                 is_can=(m.decision == 'cancelled').astype(int),
                 days_since=(m.cutoff - m.decision_datetime).dt.days)
    out = m.groupby('application_id').agg(
        prev_cnt=('decision', 'size'), prev_cnt_appr=('is_appr', 'sum'),
        prev_cnt_rej=('is_rej', 'sum'), prev_cnt_can=('is_can', 'sum'),
        prev_req_mean=('requested_amount', 'mean'), prev_appr_max=('approved_amount', 'max'),
        prev_appr_sum=('approved_amount', 'sum'), prev_rate_mean=('annual_interest_rate', 'mean'),
        prev_inst_mean=('scheduled_monthly_installment', 'mean'), prev_last_gap_days=('days_since', 'min'))
    out['prev_rej_ratio'] = out.prev_cnt_rej / out.prev_cnt
    out['prev_appr_ratio'] = out.prev_cnt_appr / out.prev_cnt
    return out

print('f_previous siap')

### 4.3 Riwayat pembayaran
Rasio keterlambatan (30/60/90 hari), rata-rata DPD maksimum, rasio tunggakan, untuk seluruh riwayat dan 180 hari terakhir.

In [ ]:
# [Fitur] Modul repayment_history
def f_repay(A, R):
    m = join_before(A, R, 'due_date')
    m = m.assign(late=(m.max_days_past_due > 0).astype(int),
                 late30=(m.max_days_past_due >= 30).astype(int),
                 late60=(m.max_days_past_due >= 60).astype(int),
                 late90=(m.max_days_past_due >= 90).astype(int),
                 paid_ratio=m.amount_paid / m.amount_due.replace(0, np.nan),
                 unpaid=(m.amount_paid < m.amount_due).astype(int),
                 days_since_due=(m.cutoff - m.due_date).dt.days)
    def block(d, p):
        return d.groupby('application_id').agg(**{
            f'{p}_cnt_inst': ('installment_id', 'size'), f'{p}_max_dpd': ('max_days_past_due', 'max'),
            f'{p}_mean_dpd': ('max_days_past_due', 'mean'), f'{p}_late_ratio': ('late', 'mean'),
            f'{p}_late30_ratio': ('late30', 'mean'), f'{p}_late60_ratio': ('late60', 'mean'),
            f'{p}_late90_ratio': ('late90', 'mean'), f'{p}_paid_ratio': ('paid_ratio', 'mean'),
            f'{p}_unpaid_ratio': ('unpaid', 'mean')})
    out = block(m, 'rep').join(block(m[m.days_since_due <= 180], 'rep6'))
    g = m.groupby('application_id')
    out['rep_cnt_loans'] = g['loan_id'].nunique()
    out['rep_due_sum'] = g['amount_due'].sum()
    out['rep_last_gap_days'] = g['days_since_due'].min()
    return out

print('f_repay siap')

### 4.4 Perilaku rekening
Saldo, arus masuk/keluar, dan hari saldo rendah dari **bulan lengkap** sebelum cutoff, seluruh riwayat dan 3 bulan terakhir.

In [ ]:
# [Fitur] Modul account_behavior
def f_account(A, B):
    B = B.assign(month_end=B['month'] + pd.offsets.MonthBegin(1))
    m = join_before(A, B, 'month_end', le=True)          # hanya bulan yang sudah selesai
    m = m.assign(months_ago=(m.cutoff.dt.year - m.month.dt.year) * 12 + (m.cutoff.dt.month - m.month.dt.month),
                 net_flow=m.inflows - m.outflows,
                 out_in_ratio=m.outflows / m.inflows.replace(0, np.nan))
    def block(d, p):
        return d.groupby('application_id').agg(**{
            f'{p}_cnt_months': ('month', 'size'), f'{p}_avg_bal': ('average_balance', 'mean'),
            f'{p}_min_bal': ('minimum_balance', 'min'), f'{p}_low_days': ('low_balance_days', 'mean'),
            f'{p}_inflow': ('inflows', 'mean'), f'{p}_inflow_std': ('inflows', 'std'),
            f'{p}_outflow': ('outflows', 'mean'), f'{p}_income_cr': ('income_like_credits', 'mean'),
            f'{p}_net_flow': ('net_flow', 'mean'), f'{p}_out_in': ('out_in_ratio', 'mean')})
    out = block(m, 'acc').join(block(m[m.months_ago <= 3], 'acc3'))
    out['acc_bal_trend'] = out.acc3_avg_bal / out.acc_avg_bal.replace(0, np.nan)
    return out

print('f_account siap')

### 4.5 Bureau kredit
Laporan terbaru per provider yang **sudah tersedia** (`available_datetime`) pada cutoff: skor, saldo, utilisasi, inquiry, dan akun menunggak.

In [ ]:
# [Fitur] Modul bureau_observations
def f_bureau(A, U):
    m = join_before(A, U, 'available_datetime', key='application_id', le=True)
    # laporan terbaru yang sudah tersedia per provider
    m = m.sort_values('report_datetime').groupby(['application_id', 'bureau_provider']).tail(1)
    out = m.groupby('application_id').agg(
        bur_cnt_reports=('bureau_provider', 'size'), bur_score_mean=('score', 'mean'),
        bur_score_min=('score', 'min'), bur_score_max=('score', 'max'),
        bur_active=('active_accounts', 'sum'), bur_outstanding=('outstanding_balance', 'sum'),
        bur_rev_limit=('revolving_credit_limit', 'sum'), bur_rev_out=('revolving_outstanding_balance', 'sum'),
        bur_util_mean=('utilization', 'mean'), bur_util_max=('utilization', 'max'),
        bur_inq=('inquiries_last_6m', 'sum'), bur_inq_max=('inquiries_last_6m', 'max'),
        bur_delinq=('delinquent_accounts_last_12m', 'sum'), bur_delinq_max=('delinquent_accounts_last_12m', 'max'))
    out['bur_util_total'] = out.bur_rev_out / out.bur_rev_limit.replace(0, np.nan)
    out['bur_score_missing_share'] = m['score'].isna().groupby(m['application_id']).mean()
    return out

print('f_bureau siap')

### 4.6 E-commerce dasar
Jumlah order, rasio pembatalan, total belanja (order dideduplikasi, order batal bukan belanja), metode bayar, dan porsi kategori.

In [ ]:
# [Fitur] Modul ecommerce_transactions
PM = ['COD', 'e_wallet', 'bank_transfer', 'debit_card', 'credit_card']
def f_ecom(A, E):
    m = join_before(A, E, 'transaction_datetime')
    m = m.assign(days_since=(m.cutoff - m.transaction_datetime).dt.days)
    o = m.drop_duplicates(['application_id', 'order_id']).copy()     # deduplikasi order_total
    o['cancel'] = (o.order_status == 'cancelled').astype(int)
    o['spend'] = np.where(o.order_status == 'completed', o.order_total, 0)
    o['spend_90d'] = np.where(o.days_since <= 90, o.spend, 0)
    for k in PM:
        o[f'pm_{k}'] = (o.payment_method == k).astype(int)
    out = o.groupby('application_id').agg(
        ec_cnt_orders=('order_id', 'size'), ec_cancel_ratio=('cancel', 'mean'),
        ec_spend_total=('spend', 'sum'), ec_spend_90d=('spend_90d', 'sum'),
        ec_order_mean=('spend', 'mean'), ec_last_gap_days=('days_since', 'min'),
        **{f'ec_pm_{k}': (f'pm_{k}', 'mean') for k in PM})
    ic = m[m.order_status == 'completed']                              # kategori hanya dari order selesai
    cat = ic.pivot_table(index='application_id', columns='product_category',
                         values='item_total', aggfunc='sum', fill_value=0)
    cat = cat.div(cat.sum(axis=1).replace(0, np.nan), axis=0).add_prefix('ec_share_')
    return out.join(cat)

print('f_ecom siap')

### 4.7 Rakit dan bangun tabel fitur `FE`
Menjalankan semua modul v0 dan menggabungkannya per `application_id`. Customer tanpa riwayat diberi flag `has_*` dan hitungan 0 (bukan dibiarkan sebagai data hilang tanpa tanda).

In [ ]:
# [Fitur] Perakitan semua modul menjadi tabel fitur FE
FIRST_COL = {'prev': 'prev_cnt', 'rep': 'rep_cnt_inst', 'acc': 'acc_cnt_months',
             'bur': 'bur_cnt_reports', 'ec': 'ec_cnt_orders'}
CAT_COLS = ['loan_purpose', 'region', 'employment_type', 'housing_status']


def assemble_features(ALL, T):
    mods = {
        'prev': f_previous(ALL, T['previous_applications']),
        'rep':  f_repay(ALL, T['repayment_history']),
        'acc':  f_account(ALL, T['account_behavior']),
        'bur':  f_bureau(ALL, T['bureau_observations']),
        'ec':   f_ecom(ALL, T['ecommerce_transactions']),
    }
    FE = ALL.set_index('application_id')[['customer_id', 'is_train', 'target', 'decision_datetime']] \
            .join(f_base(ALL, T['customer_profiles']))
    for p, mod in mods.items():
        FE = FE.join(mod, how='left')
        FE[f'has_{p}'] = FE[FIRST_COL[p]].notna().astype(int)      # flag: punya riwayat atau tidak
    cnt_cols = [c for c in FE.columns if '_cnt' in c]
    FE[cnt_cols] = FE[cnt_cols].fillna(0)
    for c in CAT_COLS:
        FE[c] = FE[c].astype('category')
    return FE


t0 = time.time()
FE = assemble_features(ALL, T)
assert FE.index.is_unique and len(FE) == len(ALL)
print(f'FE dibangun: {FE.shape} | indeks unik | {time.time() - t0:.0f}s')

## 5. Fitur v1: versi rinci + rasio lintas tabel

### 5.1 E-commerce v1
Pesanan selesai vs batal, jendela 30/90/180/365 hari, kecepatan belanja, jarak antar order, harga satuan, dan komposisi kategori.

In [ ]:
# [Fitur v1] E-commerce: pesanan selesai vs batal, jendela waktu, kategori, interval antar-order
def f_ecom_v1(A, E):
    m = join_before(A, E, 'transaction_datetime')
    m['days_since'] = (m.cutoff - m.transaction_datetime).dt.days
    m['done'] = m.order_status == 'completed'
    o = m.drop_duplicates(['application_id', 'order_id'])                 # satu baris per order
    oc, ox = o[o.done], o[~o.done]
    ga, gc, gx = o.groupby('application_id'), oc.groupby('application_id'), ox.groupby('application_id')

    out = pd.DataFrame(index=ga.size().index)
    out['ec2_cnt_done'], out['ec2_cnt_cancel'] = gc.size(), gx.size()
    n_d, n_c = out.ec2_cnt_done.fillna(0), out.ec2_cnt_cancel.fillna(0)
    out['ec2_cancel_ratio'] = n_c / (n_d + n_c)
    out['ec2_done_mean'], out['ec2_done_median'] = gc['order_total'].mean(), gc['order_total'].median()
    out['ec2_done_max'], out['ec2_done_std'] = gc['order_total'].max(), gc['order_total'].std()
    out['ec2_cancel_mean'] = gx['order_total'].mean()
    out['ec2_cancel_value_ratio'] = gx['order_total'].sum() / ga['order_total'].sum()
    out['ec2_sum_done'] = gc['order_total'].sum()
    out['ec2_age_days'] = ga['days_since'].max()
    out['ec2_last_done_gap'], out['ec2_last_cancel_gap'] = gc['days_since'].min(), gx['days_since'].min()

    for w in (30, 90, 180, 365):
        c_w, x_w = oc[oc.days_since <= w].groupby('application_id'), ox[ox.days_since <= w].groupby('application_id')
        out[f'ec2_cnt_done_{w}d'], out[f'ec2_cnt_cancel_{w}d'] = c_w.size(), x_w.size()
        out[f'ec2_sum_done_{w}d'], out[f'ec2_mean_done_{w}d'] = c_w['order_total'].sum(), c_w['order_total'].mean()
        nd, nc = out[f'ec2_cnt_done_{w}d'].fillna(0), out[f'ec2_cnt_cancel_{w}d'].fillna(0)
        out[f'ec2_cancel_ratio_{w}d'] = nc / (nd + nc)
    out['ec2_vel_30_180'] = out.ec2_sum_done_30d.fillna(0) / (out.ec2_sum_done_180d / 6).replace(0, np.nan)
    out['ec2_vel_90_365'] = out.ec2_sum_done_90d.fillna(0) / (out.ec2_sum_done_365d / 4).replace(0, np.nan)

    s = oc.sort_values(['application_id', 'days_since']).copy()
    s['gap'] = s.groupby('application_id')['days_since'].diff()           # jarak antar order selesai
    gg = s.groupby('application_id')['gap']
    out['ec2_gap_mean'], out['ec2_gap_std'], out['ec2_gap_max'] = gg.mean(), gg.std(), gg.max()

    o90 = oc[oc.days_since <= 90]
    for k in PM:
        out[f'ec2_pm90_{k}'] = (o90.payment_method == k).groupby(o90.application_id).mean()

    it = m[m.done].copy()
    it['unit_price'] = it.item_total / it.quantity.replace(0, np.nan)
    gi = it.groupby('application_id')
    out['ec2_unit_price_mean'] = gi['unit_price'].mean()
    out['ec2_items_per_order'] = gi.size() / out.ec2_cnt_done
    cat = it.pivot_table(index='application_id', columns='product_category', values='item_total', aggfunc='sum', fill_value=0)
    share = cat.div(cat.sum(axis=1).replace(0, np.nan), axis=0)
    it90 = it[it.days_since <= 90]
    cat90 = it90.pivot_table(index='application_id', columns='product_category', values='item_total', aggfunc='sum', fill_value=0)
    share90 = cat90.div(cat90.sum(axis=1).replace(0, np.nan), axis=0)
    out = out.join(cat.add_prefix('ec2_sum_cat_')).join(share.add_prefix('ec2_share_cat_')) \
             .join(cat90.add_prefix('ec2_sum90_cat_')).join(share90.add_prefix('ec2_share90_cat_'))
    out['ec2_cat_hhi'] = (share ** 2).sum(axis=1)
    out['ec2_cat_ndistinct'] = (cat > 0).sum(axis=1)
    return out

print('f_ecom_v1 siap')

### 5.2 Bureau v1
Skor per provider **dinormalisasi memakai statistik train saja**, selisih antar provider, dan tren dari laporan lama customer yang sudah tersedia sebelum cutoff.

In [ ]:
# [Fitur v1] Bureau: skor per provider (dinormalisasi), selisih antar provider, tren dari laporan lama
BUR_SHORT = {'score': 'score', 'utilization': 'util', 'delinquent_accounts_last_12m': 'delinq',
             'inquiries_last_6m': 'inq', 'outstanding_balance': 'outst'}

def f_bureau_v1(A, U, train_ids):
    # laporan terbaru yang sudah tersedia per provider untuk aplikasi ini
    m = join_before(A, U, 'available_datetime', key='application_id', le=True)
    m = m.sort_values('report_datetime').groupby(['application_id', 'bureau_provider']).tail(1).copy()
    ref = m[m.application_id.isin(train_ids)].groupby('bureau_provider')['score'].agg(['mean', 'std'])   # statistik dari train saja
    m['score_z'] = (m['score'] - m.bureau_provider.map(ref['mean'])) / m.bureau_provider.map(ref['std'])

    wide = m.pivot(index='application_id', columns='bureau_provider', values=list(BUR_SHORT))
    wide.columns = [f'bur2_{BUR_SHORT[f]}_{p[-1]}' for f, p in wide.columns]
    sc = [c for c in wide.columns if c.startswith('bur2_score_')]
    out = wide.copy()
    out['bur2_score_spread'] = wide[sc].max(axis=1) - wide[sc].min(axis=1)
    out['bur2_score_nmiss'] = wide[sc].isna().sum(axis=1)
    gz = m.groupby('application_id')['score_z']
    out['bur2_z_mean'], out['bur2_z_min'], out['bur2_z_max'], out['bur2_z_std'] = gz.mean(), gz.min(), gz.max(), gz.std()

    # laporan lama (aplikasi sebelumnya milik customer yang sama), hanya yang sudah tersedia sebelum cutoff
    a = A[['application_id', 'customer_id', 'decision_datetime']].rename(columns={'decision_datetime': 'cutoff'})
    h = a.merge(U.rename(columns={'application_id': 'src_app'}), on='customer_id')
    h = h[(h.available_datetime <= h.cutoff) & (h.src_app != h.application_id)].copy()
    h['score_z'] = (h['score'] - h.bureau_provider.map(ref['mean'])) / h.bureau_provider.map(ref['std'])
    last_t = h.groupby('application_id')['report_datetime'].transform('max')
    hl = h[h.report_datetime == last_t]
    gh, gl = h.groupby('application_id'), hl.groupby('application_id')
    hist = pd.DataFrame({'bur2_hist_cnt': gh.size(), 'bur2_hist_z_mean': gh['score_z'].mean(),
                         'bur2_hist_z_last': gl['score_z'].mean(), 'bur2_hist_util_mean': gh['utilization'].mean(),
                         'bur2_hist_delinq_max': gh['delinquent_accounts_last_12m'].max()})
    hist['bur2_hist_gap_days'] = (gl['cutoff'].first() - gl['report_datetime'].first()).dt.days
    out = out.join(hist)
    out['bur2_z_delta'] = out.bur2_z_mean - out.bur2_hist_z_last
    out['bur2_util_delta'] = out[[c for c in out.columns if c.startswith('bur2_util_')]].mean(axis=1) - out.bur2_hist_util_mean
    return out

print('f_bureau_v1 siap')

### 5.3 Repayment v1
Status saat cutoff (DPD berjalan, nominal tunggakan), telat beruntun dari cicilan terbaru, bobot recency, dan pinjaman terburuk.

In [ ]:
# [Fitur v1] Repayment: status saat cutoff (DPD saat ini, tunggakan), streak, cicilan terakhir, bobot nominal dan recency
def f_repay_v1(A, R_):
    m = join_before(A, R_, 'due_date')
    m = m.assign(late=(m.max_days_past_due > 0).astype(int), cur_late=(m.days_past_due > 0).astype(int),
                 days_since_due=(m.cutoff - m.due_date).dt.days,
                 overdue_amt=np.clip(m.amount_due - m.amount_paid, 0, None))
    m = m.sort_values(['application_id', 'due_date'], ascending=[True, False])
    m['rk'] = m.groupby('application_id').cumcount()                        # 0 = cicilan terbaru
    m['w'] = np.exp(-m.days_since_due / 180)
    m['late_amt'], m['late_rec'] = m.late * m.amount_due, m.late * m.w
    m['streak'] = m.groupby('application_id')['late'].cumprod()           # telat beruntun dari cicilan terbaru
    g = m.groupby('application_id')
    out = pd.DataFrame({
        'rep2_cur_dpd_max': g['days_past_due'].max(), 'rep2_cur_dpd_mean': g['days_past_due'].mean(),
        'rep2_cnt_overdue': g['cur_late'].sum(), 'rep2_sum_overdue_amt': g['overdue_amt'].sum(),
        'rep2_overdue_ratio': g['overdue_amt'].sum() / g['amount_due'].sum(),
        'rep2_late_amt_ratio': g['late_amt'].sum() / g['amount_due'].sum(),
        'rep2_late_rec_ratio': g['late_rec'].sum() / g['w'].sum(),
        'rep2_late_streak': g['streak'].sum(),
    })
    out['rep2_pay_gap_days'] = (g['cutoff'].first() - g['last_payment_datetime'].max()).dt.days
    for n in (3, 6):
        gn = m[m.rk < n].groupby('application_id')
        out[f'rep2_last{n}_late_ratio'], out[f'rep2_last{n}_dpd_max'] = gn['late'].mean(), gn['max_days_past_due'].max()
    pl = m.groupby(['application_id', 'loan_id']).agg(l=('late', 'mean'), mx=('max_days_past_due', 'max'))
    out['rep2_worst_loan_late'] = pl.groupby('application_id')['l'].max()
    out['rep2_cnt_loans_90'] = (pl.mx >= 90).astype(int).groupby('application_id').sum()
    return out

print('f_repay_v1 siap')

### 5.4 Previous v1
Recency pengajuan, pinjaman lama yang masih aktif saat cutoff, dan beban cicilan berjalan.

In [ ]:
# [Fitur v1] Previous: recency pengajuan, pinjaman aktif, beban kewajiban
def f_previous_v1(A, P):
    m = join_before(A, P, 'decision_datetime')
    m['days_since'] = (m.cutoff - m.decision_datetime).dt.days
    ap = m[m.decision == 'approved'].copy()
    ap['end'] = ap.decision_datetime + pd.to_timedelta(ap.term_months * 30.4, unit='D')
    act = ap[ap.end > ap.cutoff]                                            # pinjaman lama yang masih berjalan saat cutoff
    ga, gact = ap.groupby('application_id'), act.groupby('application_id')
    return pd.DataFrame({
        'prev2_first_gap_days': m.groupby('application_id')['days_since'].max(),
        'prev2_cnt_180d': m[m.days_since <= 180].groupby('application_id').size(),
        'prev2_cnt_365d': m[m.days_since <= 365].groupby('application_id').size(),
        'prev2_last_appr_gap_days': ga['days_since'].min(),
        'prev2_cnt_active': gact.size(),
        'prev2_active_inst_sum': gact['scheduled_monthly_installment'].sum(),
        'prev2_active_amt_sum': gact['approved_amount'].sum(),
        'prev2_appr_amt_max': ga['approved_amount'].max(),
        'prev2_appr_rate_max': ga['annual_interest_rate'].max(),
    })

print('f_previous_v1 siap')

### 5.5 Rakit v1 dan rasio lintas tabel, bangun `FE1`
Menggabungkan modul v1 ke `FE` dan menambah rasio lintas tabel berawalan `x_` (belanja terhadap penghasilan, total cicilan terhadap penghasilan, dan sebagainya).

In [ ]:
# [Fitur v1] Perakitan fitur v1 + rasio lintas tabel (prefix x_)
def assemble_v1(FE, ALL, T, train_ids):
    mods = [f_ecom_v1(ALL, T['ecommerce_transactions']), f_bureau_v1(ALL, T['bureau_observations'], train_ids),
            f_repay_v1(ALL, T['repayment_history']), f_previous_v1(ALL, T['previous_applications'])]
    X1 = pd.DataFrame(index=FE.index)
    for mod in mods:
        X1 = X1.join(mod, how='left')
    cnt = [c for c in X1.columns if '_cnt' in c]
    X1[cnt] = X1[cnt].fillna(0)
    inc = FE['declared_monthly_income'].replace(0, np.nan)
    outst = [c for c in X1.columns if c.startswith('bur2_outst_')]
    X1['x_ec90_to_income'] = X1['ec2_sum_done_90d'] / inc
    X1['x_ec90_to_inst'] = X1['ec2_sum_done_90d'] / FE['proposed_monthly_installment'].replace(0, np.nan)
    X1['x_ecmean_to_income'] = X1['ec2_done_mean'] / inc
    X1['x_active_inst_to_income'] = X1['prev2_active_inst_sum'].fillna(0) / inc
    X1['x_total_inst_to_income'] = (X1['prev2_active_inst_sum'].fillna(0) + FE['proposed_monthly_installment']) / inc
    X1['x_outst_to_income'] = X1[outst].sum(axis=1, min_count=1) / (inc * 12)
    X1['x_income_vs_observed'] = FE['declared_monthly_income'] / FE['acc_income_cr'].replace(0, np.nan)
    X1['x_loan_to_prev_max'] = FE['loan_amount'] / X1['prev2_appr_amt_max'].replace(0, np.nan)
    X1['x_rate_diff_prev'] = FE['annual_interest_rate'] - X1['prev2_appr_rate_max']
    return FE.join(X1)


t0 = time.time()
FE1 = assemble_v1(FE, ALL, T, set(tr.application_id))
assert FE1.index.is_unique and len(FE1) == len(ALL)
print(f'FE1 dibangun: {FE1.shape} | fitur v1 baru: {FE1.shape[1] - FE.shape[1]} | {time.time() - t0:.0f}s')

## 6. Fitur e-commerce v2 (`ec3_*`)
Lapis fitur terakhir dan paling berpengaruh pada eksperimen (KS validasi naik sekitar +0.009 pada CV berulang). Semua modul memakai satu konteks bersama yang sudah difilter `< decision_datetime`.

### 6.1 Konfigurasi
Jendela waktu, jumlah bucket tren (6 x 30 hari), daftar metode bayar dan kategori, dan daftar modul.

In [ ]:
EC3_WINDOWS = (7, 14, 30, 90, 180)    # jendela hari; 7 dan 14 dikeluarkan sebagai fitur, sisanya basis rasio lonjakan
EC3_TREND_BUCKETS = 6                 # jumlah bucket 30 hari untuk tren belanja (6 = 180 hari)
EC3_PM = ['COD', 'e_wallet', 'bank_transfer', 'debit_card', 'credit_card']
EC3_CATS = ['groceries', 'fashion', 'electronics', 'home', 'health', 'education', 'personal_care']
EC3_MODS = ['win', 'time', 'trd', 'cncl', 'prc', 'reg', 'x']   # kunci modul = bagian dari prefix ec3_<kunci>_
print(f'Konfigurasi ec3 siap: {len(EC3_MODS)} modul {EC3_MODS}')

### 6.2 Persiapan konteks bersama
Memperkecil tabel e-commerce, memfilter transaksi `< decision_datetime`, lalu memisahkannya menjadi tabel **order** (dideduplikasi) dan **item**. Ambang statistik (median harga per kategori, kuantil order) dihitung dari **aplikasi train saja tanpa target**.

In [ ]:
def ec3_slim(E):
    """Ambil kolom yang dipakai dan perkecil dtype."""
    e = E[['customer_id', 'order_id', 'transaction_datetime', 'product_category', 'quantity',
           'item_total', 'order_total', 'payment_method', 'order_status']].copy()
    e['order_id'] = pd.factorize(e['order_id'])[0].astype('int32')
    e['quantity'] = e['quantity'].astype('int32')
    for c in ('product_category', 'payment_method', 'order_status'):
        e[c] = e[c].astype('category')
    return e


def ec3_prepare(A, E):
    """Filter transaksi < cutoff, hitung umur transaksi (hari), pisahkan tabel order dan item."""
    A2 = A[['application_id', 'customer_id', 'decision_datetime']].copy()
    A2['application_id'] = A2['application_id'].astype('category')
    m = join_before(A2, E, 'transaction_datetime')                        # strict: transaksi < decision_datetime
    m['days'] = ((m['cutoff'] - m['transaction_datetime']).dt.total_seconds() / 86400).astype('float32')
    m['done'] = (m['order_status'] == 'completed').to_numpy()
    o = (m.drop_duplicates(['application_id', 'order_id'])             # order_total diulang per item
          [['application_id', 'order_id', 'transaction_datetime', 'days', 'done', 'order_total', 'payment_method']]
          .reset_index(drop=True))
    it = m[['application_id', 'days', 'done', 'product_category', 'quantity', 'item_total']].reset_index(drop=True)
    return o, it


def ec3_thresholds(o, it, train_ids):
    """Ambang global dari aplikasi train saja (tanpa target): median harga satuan per kategori, kuantil 90 order."""
    ids = list(train_ids)
    d = it[it['application_id'].isin(ids) & it['done'] & (it['quantity'] > 0)]
    up = d['item_total'] / d['quantity']
    cat_med = up.groupby(d['product_category'], observed=True).median().to_dict()
    od = o[o['application_id'].isin(ids) & o['done']]
    return {'cat_med': cat_med, 'ord_q90': float(od['order_total'].quantile(0.9))}


def ec3_context(ALL, E, train_ids):
    """Konteks bersama yang dipakai semua modul."""
    o, it = ec3_prepare(ALL, ec3_slim(E))
    return {'o': o, 'it': it, 'thr': ec3_thresholds(o, it, train_ids)}


def _fin(df):
    """Rapikan keluaran modul: float32 dan index string application_id."""
    df = df.astype('float32')
    df.index = df.index.astype(str)
    df.index.name = 'application_id'
    return df

print('Fungsi persiapan konteks ec3 siap')

### 6.3 Modul `win`: jendela pendek dan lonjakan
Jumlah order selesai/batal dan belanja pada 7 dan 14 hari terakhir, plus rasio lonjakan terhadap laju normal.

In [ ]:
def ec3_win(ctx, ws=EC3_WINDOWS):
    o = ctx['o']
    d, tot = o['done'].to_numpy(), o['order_total'].to_numpy()
    cols = {}
    for w in ws:                                                        # satu groupby untuk semua jendela
        inw = (o['days'] <= w).to_numpy()
        cols[f'cd_{w}'] = (inw & d).astype('float32')
        cols[f'cx_{w}'] = (inw & ~d).astype('float32')
        cols[f'sd_{w}'] = np.where(inw & d, tot, 0.0)
    g = pd.DataFrame(cols, index=o.index).groupby(o['application_id'], observed=True).sum()

    out = pd.DataFrame(index=g.index)
    for w in (7, 14):
        out[f'ec3_win_cnt_done_{w}d'] = g[f'cd_{w}']
        out[f'ec3_win_cnt_cancel_{w}d'] = g[f'cx_{w}']
        out[f'ec3_win_sum_done_{w}d'] = g[f'sd_{w}']
        out[f'ec3_win_cancel_ratio_{w}d'] = g[f'cx_{w}'] / (g[f'cd_{w}'] + g[f'cx_{w}']).replace(0, np.nan)
    nz = lambda s: s.replace(0, np.nan)
    out['ec3_win_spk_sum_7_30'] = g['sd_7'] / nz(g['sd_30'] * 7 / 30)
    out['ec3_win_spk_sum_14_90'] = g['sd_14'] / nz(g['sd_90'] * 14 / 90)
    out['ec3_win_spk_sum_30_180'] = g['sd_30'] / nz(g['sd_180'] * 30 / 180)
    out['ec3_win_spk_ord_7_30'] = g['cd_7'] / nz(g['cd_30'] * 7 / 30)
    out['ec3_win_spk_ord_14_90'] = g['cd_14'] / nz(g['cd_90'] * 14 / 90)
    cr = lambda w: g[f'cx_{w}'] / nz(g[f'cd_{w}'] + g[f'cx_{w}'])
    out['ec3_win_spk_cancel_30_180'] = cr(30) / nz(cr(180))
    return _fin(out)

print('ec3_win siap')

### 6.4 Modul `time`: pola waktu transaksi
Porsi order malam, akhir pekan, dan sekitar gajian; jam rata-rata; jam sejak order terakhir.

In [ ]:
def ec3_time(ctx):
    o = ctx['o']
    t = o['transaction_datetime']
    h, dow, dom = t.dt.hour.to_numpy(), t.dt.dayofweek.to_numpy(), t.dt.day.to_numpy()
    ang = 2 * np.pi * h / 24
    base = pd.DataFrame({'night': h < 6, 'late': h >= 20, 'wkend': dow >= 5,
                         'payday': (dom >= 25) | (dom <= 5)}, index=o.index).astype('float32')
    key = o['application_id']
    g_all = base.assign(hsin=np.sin(ang), hcos=np.cos(ang)).groupby(key, observed=True).mean().add_prefix('ec3_time_')
    r = (o['days'] <= 90).to_numpy()
    g_90 = base[r].groupby(key[r], observed=True).mean().add_prefix('ec3_time_').add_suffix('_90d')
    out = g_all.join(g_90)
    out['ec3_time_last_order_h'] = o.groupby('application_id', observed=True)['days'].min() * 24
    return _fin(out)

print('ec3_time siap')

### 6.5 Modul `trd`: tren belanja bulanan
Kemiringan dan variasi belanja per bucket 30 hari, jumlah bucket kosong, dan rasio bulan terakhir terhadap sebelumnya.

In [ ]:
def ec3_trd(ctx, nb=EC3_TREND_BUCKETS):
    o = ctx['o']
    od = o[o['done'] & (o['days'] < 30 * nb)]
    b = np.floor(od['days'].to_numpy() / 30).astype('int8')            # 0 = 30 hari terakhir
    gb = od.groupby([od['application_id'], b], observed=True)
    S = gb['order_total'].sum().unstack(fill_value=0).reindex(columns=range(nb), fill_value=0)
    N = gb.size().unstack(fill_value=0).reindex(columns=range(nb), fill_value=0)
    idx = S.index
    S, N = S.to_numpy('float64'), N.to_numpy('float64')

    x = -(np.arange(nb) - (nb - 1) / 2)                                 # positif = naik menuju cutoff
    slope = lambda M: (M @ x) / (x ** 2).sum()
    mean, nz = S.mean(axis=1), lambda a: np.where(a == 0, np.nan, a)
    out = pd.DataFrame({
        'ec3_trd_slope_amt': slope(S),
        'ec3_trd_slope_amt_rel': slope(S) / nz(mean),
        'ec3_trd_slope_ord': slope(N),
        'ec3_trd_cv_amt': S.std(axis=1) / nz(mean),
        'ec3_trd_zero_buckets': (S == 0).sum(axis=1),
        'ec3_trd_recent_vs_prior': S[:, 0] / nz(S[:, 1:].mean(axis=1)),
        'ec3_trd_last3_vs_prev3': S[:, :nb // 2].sum(axis=1) / nz(S[:, nb // 2:].sum(axis=1)),
    }, index=idx)
    return _fin(out)

print('ec3_trd siap')

### 6.6 Modul `cncl`: pembatalan
Jumlah dan rasio pembatalan per metode bayar dan per kategori produk, serta porsi COD di antara pembatalan.

In [ ]:
def _wide(g, cols):
    w = g.unstack()
    w.columns = w.columns.astype(str)
    return w.reindex(columns=cols)


def ec3_cncl(ctx):
    o, it = ctx['o'], ctx['it']
    ox = o.assign(cx=(~o['done']).astype('float32'))
    gp = ox.groupby(['application_id', 'payment_method'], observed=True)['cx'].agg(['size', 'sum'])
    n, c = _wide(gp['size'], EC3_PM), _wide(gp['sum'], EC3_PM)
    out = pd.DataFrame(index=n.index)
    for k in EC3_PM:
        out[f'ec3_cncl_cnt_{k}'] = c[k]
        out[f'ec3_cncl_ratio_{k}'] = c[k] / n[k].replace(0, np.nan)
    out['ec3_cncl_cod_share'] = c['COD'] / c.sum(axis=1).replace(0, np.nan)

    ix = it.assign(cx=(~it['done']).astype('float32'))
    gc = ix.groupby(['application_id', 'product_category'], observed=True)['cx'].agg(['size', 'sum'])
    n2, c2 = _wide(gc['size'], EC3_CATS), _wide(gc['sum'], EC3_CATS)
    ratio = (c2 / n2.replace(0, np.nan)).add_prefix('ec3_cncl_cat_ratio_')
    return _fin(out.join(ratio))

print('ec3_cncl siap')

### 6.7 Modul `prc`: indeks harga
Harga satuan dibagi median kategorinya (menghilangkan efek komposisi kategori), indeks per kategori, dan porsi order bernilai tinggi.

In [ ]:
def ec3_prc(ctx):
    it, o, thr = ctx['it'], ctx['o'], ctx['thr']
    d = it[it['done'] & (it['quantity'] > 0)]
    up = (d['item_total'] / d['quantity']).to_numpy('float64')
    med = d['product_category'].map(thr['cat_med']).astype('float64').to_numpy()
    d = d.assign(up=up, idx=up / med)
    g = d.groupby('application_id', observed=True)
    out = g.agg(ec3_prc_idx_mean=('idx', 'mean'), ec3_prc_idx_std=('idx', 'std'), ec3_prc_idx_max=('idx', 'max'),
                ec3_prc_up_mean=('up', 'mean'), ec3_prc_up_std=('up', 'std'),
                ec3_prc_qty_mean=('quantity', 'mean'), ec3_prc_qty_max=('quantity', 'max'))
    out['ec3_prc_up_cv'] = out['ec3_prc_up_std'] / out['ec3_prc_up_mean'].replace(0, np.nan)
    out['ec3_prc_idx_p90'] = g['idx'].quantile(0.9)

    cat_idx = (d.groupby(['application_id', 'product_category'], observed=True)['idx'].mean()
                 .pipe(_wide, EC3_CATS).add_prefix('ec3_prc_cat_idx_'))
    od = o[o['done']]
    hv = od['order_total'] >= thr['ord_q90']
    k = od['application_id']
    out['ec3_prc_hv_share'] = hv.groupby(k, observed=True).mean()
    out['ec3_prc_hv_cnt_90d'] = (hv & (od['days'] <= 90)).groupby(k, observed=True).sum()
    return _fin(out.join(cat_idx))

print('ec3_prc siap')

### 6.8 Modul `reg`: keteraturan ritme belanja
Statistik jeda antar order, hari aktif, order per bulan, dan belanja berbobot peluruhan (half-life 30/90 hari).

In [ ]:
def ec3_reg(ctx):
    o = ctx['o']
    od = o[o['done']].sort_values(['application_id', 'days'], kind='stable')
    od = od.assign(gap=od.groupby('application_id', observed=True)['days'].diff(),
                   day=np.floor(od['days'].to_numpy()))
    g = od.groupby('application_id', observed=True)
    gg = od.groupby('application_id', observed=True)['gap']
    gap_mean, gap_std = gg.mean(), gg.std()
    nz = lambda s: s.replace(0, np.nan)

    out = pd.DataFrame({'ec3_reg_gap_median': gg.median(), 'ec3_reg_gap_min': gg.min(),
                        'ec3_reg_gap_cv': gap_std / nz(gap_mean)})
    last, age = g['days'].min(), g['days'].max()
    out['ec3_reg_last_gap_ratio'] = last / nz(gap_mean)
    per_day = od.groupby(['application_id', 'day'], observed=True).size()
    out['ec3_reg_active_days'] = per_day.groupby(level=0, observed=True).size()
    out['ec3_reg_max_orders_day'] = per_day.groupby(level=0, observed=True).max()
    out['ec3_reg_orders_per_month'] = g.size() / (age / 30).clip(lower=1)
    out['ec3_reg_recent_share_30d'] = (od['days'] <= 30).groupby(od['application_id'], observed=True).mean()
    for hl in (30, 90):                                                 # belanja berbobot peluruhan eksponensial
        w = np.exp(-od['days'].to_numpy() * np.log(2) / hl) * od['order_total'].to_numpy()
        out[f'ec3_reg_decay_sum_{hl}'] = pd.Series(w, index=od.index).groupby(od['application_id'], observed=True).sum()
    return _fin(out)

print('ec3_reg siap')

### 6.9 Modul `x`: rasio lintas tabel
Belanja jangka pendek terhadap penghasilan dan cicilan yang diajukan.

In [ ]:
def ec3_cross(X, FE):
    inc = FE['declared_monthly_income'].reindex(X.index).replace(0, np.nan)
    inst = FE['proposed_monthly_installment'].reindex(X.index).replace(0, np.nan)
    pairs = {'ec3_x_sum7_to_inc': ('ec3_win_sum_done_7d', inc), 'ec3_x_sum14_to_inc': ('ec3_win_sum_done_14d', inc),
             'ec3_x_sum14_to_inst': ('ec3_win_sum_done_14d', inst), 'ec3_x_decay30_to_inc': ('ec3_reg_decay_sum_30', inc),
             'ec3_x_decay90_to_inc': ('ec3_reg_decay_sum_90', inc), 'ec3_x_decay90_to_inst': ('ec3_reg_decay_sum_90', inst)}
    for name, (col, den) in pairs.items():
        if col in X.columns:
            X[name] = (X[col] / den).astype('float32')
    return X

print('ec3_cross siap')

### 6.10 Rakit dan bangun fitur ec3 (`X3`)
Menjalankan semua modul lewat registry (tiap modul menerima konteks yang sama), menambah fitur lintas tabel, mengganti nilai tak hingga dengan kosong, dan mengisi hitungan tanpa data dengan 0. Waktu per modul dicetak.

In [ ]:
EC3_MODULES = {'win': ec3_win, 'time': ec3_time, 'trd': ec3_trd, 'cncl': ec3_cncl, 'prc': ec3_prc, 'reg': ec3_reg}


def assemble_ecom_v2(FE, ALL, T, train_ids, modules=EC3_MODULES):
    ctx = ec3_context(ALL, T['ecommerce_transactions'], train_ids)
    X = pd.DataFrame(index=FE.index)
    for key, fn in modules.items():
        t0 = time.time()
        part = fn(ctx)
        X = X.join(part, how='left')
        print(f'  modul {key:5s}: {part.shape[1]:3d} kolom  ({time.time() - t0:.0f}s)')
    X = ec3_cross(X, FE)
    X = X.replace([np.inf, -np.inf], np.nan)
    cnt = [c for c in X.columns if '_cnt_' in c]                         # hitungan tanpa data = 0, bukan NaN
    X[cnt] = X[cnt].fillna(0)
    return X


def ec3_cols(mods):
    return [c for c in X3.columns if any(c.startswith(f'ec3_{m}_') for m in mods)]


t0 = time.time()
X3 = assemble_ecom_v2(FE, ALL, T, set(tr.application_id))
assert X3.index.equals(FE.index) and X3.index.is_unique
print(f'X3 dibangun: {X3.shape} | indeks sama dengan FE | {time.time() - t0:.0f}s')

### 6.11 Sanity check fitur ec3
Dua pengecekan sebelum dipakai model: (1) **drift train vs test** per fitur (harus kecil karena split acak; angka besar menandakan fitur dibangun tidak simetris atau bocor) dan (2) **KS univariat terhadap target** (fitur dengan KS sangat tinggi dicurigai bocor). Notebook berhenti jika drift melebihi batas (0.05 pada data asli).

In [ ]:
def ec3_sanity(X3, tr_ids, y_tr):
    is_tr = X3.index.isin(tr_ids)
    rep = pd.DataFrame({'nan_%': X3.isna().mean() * 100, 'nunique': X3.nunique()})
    rep['drift_ks'] = {c: ks_2samp(X3.loc[is_tr, c].dropna(), X3.loc[~is_tr, c].dropna()).statistic for c in X3.columns}
    yy = y_tr.reindex(X3.index[is_tr])
    rep['ks_target'] = {c: ks_2samp(X3.loc[is_tr, c][yy == 1].dropna(), X3.loc[is_tr, c][yy == 0].dropna()).statistic
                        for c in X3.columns}
    return rep.round(4)



y_tr = tr.set_index('application_id')['target']
san = ec3_sanity(X3, set(tr.application_id), y_tr)
n_const = int((san['nunique'] <= 1).sum())
print(f"drift maks = {san.drift_ks.max():.4f} ({san.drift_ks.idxmax()}) | KS target maks = {san.ks_target.max():.4f} ({san.ks_target.idxmax()}) | kolom konstan = {n_const}")
display(san.sort_values('ks_target', ascending=False).head(10))
n_tr, n_te = len(tr), len(te)
thr = max(0.05, 3 * np.sqrt((n_tr + n_te) / (n_tr * n_te)))     # batas disesuaikan dengan ukuran sampel (data asli: 0.05)
assert san.drift_ks.max() < thr, f'Drift train/test terlalu besar (> {thr:.3f}): periksa pembangunan fitur'
print(f'Sanity ec3 lolos (drift maks {san.drift_ks.max():.4f} < batas {thr:.3f})')

### 6.12 Gabungkan fitur ec3 ke `FE1`
Menempelkan semua kolom `ec3_*` ke tabel fitur akhir.

In [ ]:
FE1 = FE1.join(X3[ec3_cols(EC3_MODS)])
assert FE1.index.is_unique and len(FE1) == len(ALL)
print(f"FE1 + ec3: {FE1.shape} | kolom ec3: {len(ec3_cols(EC3_MODS))}")

## 7. Model

### 7.1 Metrik KS dan cross-validation
`ks_score` memakai `scipy.stats.ks_2samp` (selisih maksimum CDF skor kelas 1 vs 0, sama dengan evaluasi kompetisi). `train_cv` memakai `StratifiedKFold` dari scikit-learn untuk membagi data menjadi 5 fold, melatih LightGBM dengan early stopping pada AUC validasi di tiap fold, dan mengembalikan prediksi **out-of-fold** (OOF) untuk mengukur KS serta rata-rata prediksi test dari kelima model.

In [ ]:
def ks_score(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(ks_2samp(y_pred[y_true == 1], y_pred[y_true == 0]).statistic)


def train_cv(X, y, X_test, params=LGB_PARAMS, seed=SEED):
    oof, test_pred, models = np.zeros(len(X)), np.zeros(len(X_test)), []
    for k, (i_tr, i_va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(X, y)):
        m = lgb.LGBMClassifier(**params, random_state=seed + k)
        m.fit(X.iloc[i_tr], y.iloc[i_tr], eval_set=[(X.iloc[i_va], y.iloc[i_va])], eval_metric='auc',
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
        oof[i_va] = m.predict_proba(X.iloc[i_va])[:, 1]
        test_pred += m.predict_proba(X_test)[:, 1] / N_FOLDS
        models.append(m)
    return oof, test_pred, models


print('KS dan cross-validation siap')

### 7.2 Validasi dan pembuatan submission
`validate_submission` memeriksa format yang disyaratkan panitia (kolom persis `application_id,default_probability`, ID lengkap dan unik, nilai finite di rentang 0-1). `make_submission` membuat file mengikuti urutan `sample_submission` dan langsung memvalidasinya.

In [ ]:
# [Model] Fungsi validasi format submission
def validate_submission(sub, sample):
    assert list(sub.columns) == ['application_id', 'default_probability'], 'kolom salah'
    assert len(sub) == len(sample), f'jumlah baris {len(sub)} != {len(sample)}'
    assert sub.application_id.is_unique, 'application_id duplikat'
    assert set(sub.application_id) == set(sample.application_id), 'ID tidak cocok dengan sample_submission'
    p = sub.default_probability.to_numpy()
    assert np.isfinite(p).all(), 'ada NaN/inf'
    assert ((p >= 0) & (p <= 1)).all(), 'nilai di luar 0-1'
    return True


# [Model] Fungsi buat file submission
def make_submission(test_index, pred, sample, path):
    sub = pd.DataFrame({'application_id': test_index, 'default_probability': np.clip(pred, 0, 1)})
    sub = sample[['application_id']].merge(sub, on='application_id', how='left')   # ikut urutan sample
    validate_submission(sub, sample)
    sub.to_csv(path, index=False)
    return sub

print('Fungsi submission siap')

### 7.3 Bangun `X`, `y`, `X_test`
Kolom non-fitur (ID, target, cutoff), grup tanpa kontribusi (`NOISE_PREFIX`), dan kolom konstan dibuang; baris train dan test dipisahkan dengan indeks boolean pandas. ID test dicocokkan lagi dengan `sample_submission`.

In [ ]:
FEATS = [c for c in FE1.columns if c not in BASE_DROP + NOISE_COLS and not c.startswith(NOISE_PREFIX)]
is_train = FE1['is_train'] == 1
X, y, X_test = FE1.loc[is_train, FEATS], FE1.loc[is_train, 'target'].astype(int), FE1.loc[~is_train, FEATS]

print(f'train: {X.shape}  test: {X_test.shape}  target rate: {y.mean():.2%}  jumlah fitur: {len(FEATS)}')
print(f'  fitur ec3 dalam model: {sum(c.startswith("ec3_") for c in FEATS)} | kolom kategorikal: {list(X.select_dtypes("category").columns)}')
assert set(X_test.index) == set(SUB.application_id), 'ID test di FE tidak cocok dengan sample_submission'
print('ID test cocok dengan sample_submission | tidak ada kolom ID/target/cutoff di fitur')

## 8. Latih dan validasi: 5 seed x 5 fold
Setiap seed menghasilkan pembagian fold sendiri. Untuk tiap seed dicetak **OOF KS** (estimasi jujur performa pada data yang tidak dilihat model) dan AUC. Prediksi akhir adalah **rata-rata semua seed**, karena KS antar seed bervariasi cukup besar pada data berisik ini dan rata-rata menurunkan variansinya.

In [ ]:
t0 = time.time()
oofs, tests, all_models = [], [], {}
for s in BAG_SEEDS:
    oof, test_pred, models = train_cv(X, y, X_test, seed=s)
    oofs.append(oof); tests.append(test_pred); all_models[s] = models
    print(f"  seed {s}: OOF KS = {ks_score(y, oof):.4f} | AUC = {roc_auc_score(y, oof):.4f} | iterasi per fold = {[m.best_iteration_ for m in models]}")

oof_bag, test_bag = np.mean(oofs, axis=0), np.mean(tests, axis=0)
ks_bag = ks_score(y, oof_bag)
print(f'Bagging selesai: KS OOF rata-rata {len(BAG_SEEDS)} seed = {ks_bag:.4f} | AUC = {roc_auc_score(y, oof_bag):.4f} | {(time.time() - t0) / 60:.1f} menit')

## 9. Submission
Menyimpan prediksi test (rata-rata seed) sebagai file submission, divalidasi otomatis oleh `make_submission`. File bernama berisi KS OOF disimpan di folder `submissions/`, dan salinannya sebagai `submission.csv`.

In [ ]:
sub_path = f'{SUB_DIR}/sub_lgbm_ec3_bag{len(BAG_SEEDS)}_ks{ks_bag:.4f}.csv'
sub = make_submission(X_test.index, test_bag, SUB, sub_path)
sub.to_csv('submission.csv', index=False)
print(f'Submission disimpan: {sub_path} {sub.shape}')
display(sub.head())

### 9.1 Validasi akhir file submission
Membaca kembali file dari disk dan mengecek semua syarat panitia: kolom persis `application_id,default_probability`, jumlah baris sama dengan test, ID unik dan lengkap, nilai finite di rentang 0-1.

In [ ]:
chk = pd.read_csv(sub_path)
validate_submission(chk, SUB)
p = chk.default_probability
print(f'Submission valid dan siap diunggah ke Kaggle')
print(f'  baris: {len(chk):,} | ID unik: {chk.application_id.is_unique} | probabilitas: min={p.min():.4f}, median={p.median():.4f}, max={p.max():.4f}')
print(f'  file: {sub_path}')

## 10. Model pendamping: regresi logistik
Pada eksperimen, regresi logistik dengan fitur ber-kuantil menyamai LightGBM (KS 0.5115 vs 0.5121 pada seed 42) dan korelasi peringkatnya dengan LightGBM hanya 0.962, jauh lebih rendah dari antar-model pohon (0.99). Karena itu model ini dipakai sebagai pendamping untuk blend.

### 10.1 Pipeline regresi logistik
Seluruh prapemrosesan memakai `Pipeline` dan `ColumnTransformer` scikit-learn: kolom numerik diubah ke distribusi normal dengan `QuantileTransformer` (kebal terhadap outlier dan perbedaan skala), nilai kosong diisi 0 dengan indikator missing, kolom kategorikal di-one-hot, lalu `LogisticRegression`. Nilai `C = 0.03` dipilih pada eksperimen sebelumnya (KS tertinggi dari 0.01, 0.03, 0.1). OOF dihitung dengan `cross_val_predict` memakai fold yang sama dengan LightGBM pada seed yang sama, dan imputer serta transformer ikut dilatih di dalam fold sehingga tidak ada kebocoran antar fold.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer

LR_C = 0.03
CAT_COLS = list(X.select_dtypes('category').columns)
NUM_COLS = [c for c in X.columns if c not in CAT_COLS]


def make_lr(C):
    num = make_pipeline(QuantileTransformer(n_quantiles=200, output_distribution='normal', subsample=200_000, random_state=SEED),
                        SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True))
    prep = ColumnTransformer([('num', num, NUM_COLS), ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)])
    return make_pipeline(prep, LogisticRegression(C=C, max_iter=2000))


def lr_oof(C, seed):
    cv = StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed)      # fold sama dengan LightGBM pada seed ini
    return cross_val_predict(make_lr(C), X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]


print(f'Pipeline regresi logistik siap | C = {LR_C} | kolom numerik: {len(NUM_COLS)} | kategorikal: {len(CAT_COLS)}')

### 10.2 OOF dan prediksi test regresi logistik
OOF dihitung pada kelima seed lalu dirata-ratakan (seperti bagging pada LightGBM). Prediksi test berasal dari satu model yang dilatih pada seluruh data train. Dicetak juga korelasi peringkat dengan LightGBM bag: semakin rendah, semakin besar potensi blend.

In [ ]:
t0 = time.time()
oof_lr = np.mean([lr_oof(LR_C, s) for s in BAG_SEEDS], axis=0)
test_lr = make_lr(LR_C).fit(X, y).predict_proba(X_test)[:, 1]

rho = pd.Series(oof_bag).corr(pd.Series(oof_lr), method='spearman')
print(f'Regresi logistik (rata-rata {len(BAG_SEEDS)} seed): KS OOF = {ks_score(y, oof_lr):.4f} | AUC = {roc_auc_score(y, oof_lr):.4f} | {(time.time() - t0) / 60:.1f} menit')
print(f'Korelasi peringkat OOF dengan LightGBM bag: {rho:.3f}')

## 11. Blend dan tes tegas
Blend = `0.7 x peringkat(LightGBM bag) + 0.3 x peringkat(regresi logistik)`. KS hanya bergantung pada urutan, jadi blend memakai peringkat. Bobot 0.3 adalah titik tengah dataran 0.1-0.5 pada eksperimen (semua bobot memberi kenaikan positif), bukan hasil pencarian di data, sehingga tidak ada seleksi bobot yang membuat hasilnya optimistis. Tes tegas: bootstrap berpasangan 200 kali atas selisih KS blend terhadap LightGBM bag, lengkap dengan interval kepercayaan 95%.

In [ ]:
W_LR = 0.3
rank01 = lambda a: pd.Series(np.asarray(a)).rank(pct=True).to_numpy()
blend = lambda w, a, b: (1 - w) * rank01(a) + w * rank01(b)

y_arr = y.to_numpy()
r_ref, r_bl = rank01(oof_bag), blend(W_LR, oof_bag, oof_lr)
ks_blend, delta = ks_score(y_arr, r_bl), ks_score(y_arr, r_bl) - ks_score(y_arr, r_ref)

rng, n, d = np.random.default_rng(0), len(y_arr), []
for _ in range(200):
    i = rng.integers(0, n, n)
    d.append(ks_score(y_arr[i], r_bl[i]) - ks_score(y_arr[i], r_ref[i]))
lo, hi = np.percentile(d, [2.5, 97.5])

print(f'KS LightGBM bag   : {ks_score(y_arr, r_ref):.4f}')
print(f'KS blend (w={W_LR}) : {ks_blend:.4f}  (selisih {delta:+.4f})')
print(f'Bootstrap 200x    : selisih rata-rata {np.mean(d):+.4f} | interval 95% [{lo:+.4f}, {hi:+.4f}] | peluang selisih > 0: {np.mean(np.array(d) > 0):.0%}')
print('Batas bawah interval di atas nol: kenaikan terbukti pada data ini' if lo > 0
      else 'Interval mencakup nol: kenaikan belum terbukti (blend tetap menjadi kandidat pendamping)')

## 12. Submission blend
Menggabungkan peringkat prediksi test LightGBM bag dan regresi logistik dengan bobot yang sama, lalu menyimpannya melalui `make_submission` (urutan mengikuti `sample_submission`, validasi format otomatis). File LightGBM saja (bagian 9) tetap tersimpan terpisah di folder yang sama.

In [ ]:
test_blend = blend(W_LR, test_bag, test_lr)
blend_path = f'{SUB_DIR}/sub_blend_lgbm_lr_w{W_LR}_ks{ks_blend:.4f}.csv'
sub_blend = make_submission(X_test.index, test_blend, SUB, blend_path)

rho_test = pd.Series(test_blend).corr(pd.Series(rank01(test_bag)), method='spearman')
print(f'Submission blend disimpan: {blend_path} {sub_blend.shape}')
print(f'Korelasi peringkat prediksi test blend vs LightGBM bag: {rho_test:.3f}')
display(sub_blend.head())

### 12.1 Validasi akhir kedua file submission
Membaca kembali kedua file dari disk dan mengecek semua syarat panitia: kolom persis `application_id,default_probability`, ID unik dan lengkap, nilai finite di rentang 0-1.

In [ ]:
for name, path in (('LightGBM bag', sub_path), ('Blend', blend_path)):
    chk = pd.read_csv(path)
    validate_submission(chk, SUB)
    p = chk.default_probability
    print(f'{name:<13s}: valid | baris {len(chk):,} | ID unik {chk.application_id.is_unique} | min {p.min():.4f}, median {p.median():.4f}, max {p.max():.4f}')
    print(f'               {path}')

## 13. Tahap peningkatan akhir: konfigurasi
Bagian ini menguji tiga langkah yang belum pernah diuji, masing-masing dengan **aturan keputusan yang sama**: perubahan diterima jika selisih KS rata-rata berpasangan (seed dan fold sama) > 0.002 **dan** > 2x simpangan baku selisihnya.

1. **10 seed LightGBM** (bagging lebih banyak): murni pengurangan varians, tidak ada pemilihan model sehingga risikonya rendah.
2. **Pemangkasan fitur bersarang**: di tiap fold, importance (gain) dihitung dari model yang dilatih hanya pada data latih fold itu, lalu hanya K fitur teratas dipakai untuk melatih ulang. Tidak ada kebocoran ke fold validasi. Tujuannya membuang fitur redundan dari 299 fitur.
3. **Kurva belajar dan refit seluruh data**: prediksi test saat ini berasal dari model yang masing-masing hanya melihat 80% data. Kurva belajar (KS saat dilatih pada 64% vs 80% data) memberi bukti tidak langsung apakah memakai 100% data akan membantu.

Setiap langkah punya saklar dan dapat dilewati.

In [ ]:
EXTRA_SEEDS = (47, 48, 49, 50, 51)
PRUNE_K_GRID = (80, 120, 180)                 # jumlah fitur teratas yang dipertahankan
PRUNE_SCREEN, PRUNE_CONFIRM = (42, 43), (44, 45, 46)
ACCEPT_MIN, ACCEPT_K = 0.002, 2.0             # diterima jika delta rata-rata > 0.002 dan > 2x simpangan baku selisih
LC_SEEDS, LC_FRAC = (42, 43), 0.8             # kurva belajar: latih pada 80% dari data latih tiap fold
REFIT_SEEDS, REFIT_ITER_FACTOR = tuple(BAG_SEEDS), 1.1   # refit: jumlah pohon = rata-rata iterasi terbaik x 1.1

RUN_EXTRA_SEEDS = True
RUN_PRUNE = True
RUN_REFIT = True

per_seed_ks = {s: ks_score(y, o) for s, o in zip(BAG_SEEDS, oofs)}      # KS baseline per seed (pembanding berpasangan)


def compare(new_ks, seeds):
    d = np.array([new_ks[s] - per_seed_ks[s] for s in seeds])
    sd = d.std(ddof=1) if len(d) > 1 else np.nan
    return {'delta': d.mean(), 'delta_sd': sd, 'per_seed': np.round(d, 4).tolist(),
            'diterima': bool(d.mean() > ACCEPT_MIN and (np.isnan(sd) or d.mean() > ACCEPT_K * sd))}


print('Konfigurasi tahap peningkatan siap')
print('  KS baseline per seed:', {s: round(v, 4) for s, v in per_seed_ks.items()})

## 14. Langkah 1: 10 seed LightGBM
Lima seed tambahan dilatih dengan konfigurasi yang sama persis (fold berbeda per seed). Lima seed pertama tetap yang lama, jadi hasilnya identik dengan bagian 8 untuk seed 42-46. Kurva KS terhadap jumlah seed menunjukkan apakah bagging sudah jenuh.

In [ ]:
oofs10, tests10 = list(oofs), list(tests)
if RUN_EXTRA_SEEDS:
    for s in EXTRA_SEEDS:
        t0 = time.time()
        oof, test_pred, models = train_cv(X, y, X_test, seed=s)
        oofs10.append(oof); tests10.append(test_pred); all_models[s] = models
        print(f"  seed {s}: OOF KS = {ks_score(y, oof):.4f} | AUC = {roc_auc_score(y, oof):.4f} | iterasi per fold = {[m.best_iteration_ for m in models]} | {(time.time() - t0) / 60:.1f} menit")

oof_bag10, test_bag10 = np.mean(oofs10, axis=0), np.mean(tests10, axis=0)
curve = pd.Series({m: ks_score(y, np.mean(oofs10[:m], axis=0)) for m in range(1, len(oofs10) + 1)}, name='KS OOF bagging')
display(curve.to_frame().round(4).T)
print(f'Bagging {len(oofs10)} seed: KS OOF = {ks_score(y, oof_bag10):.4f} | bagging 5 seed: {ks_score(y, oof_bag):.4f} | selisih {ks_score(y, oof_bag10) - ks_score(y, oof_bag):+.4f}')

## 15. Langkah 2: pemangkasan fitur bersarang
Dari model fold yang sudah ada (dilatih hanya pada data latih fold masing-masing), importance gain menentukan K fitur teratas; model dilatih ulang pada K fitur itu dengan early stopping seperti biasa. Karena importance dihitung tanpa melihat fold validasi, KS-nya jujur. Tahap penyaringan memakai seed 42-43, lalu K terbaik dikonfirmasi pada seed 44-46 yang belum dipakai memilih.

In [ ]:
def pruned_cv(K, seed):
    oof, test_pred = np.zeros(len(X)), np.zeros(len(X_test))
    for k, (i_tr, i_va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(X, y)):
        gain = all_models[seed][k].booster_.feature_importance(importance_type='gain')
        top = set(X.columns[np.argsort(gain)[::-1][:K]])
        cols = [c for c in X.columns if c in top]
        m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed + k)
        m.fit(X.iloc[i_tr][cols], y.iloc[i_tr], eval_set=[(X.iloc[i_va][cols], y.iloc[i_va])], eval_metric='auc',
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
        oof[i_va] = m.predict_proba(X.iloc[i_va][cols])[:, 1]
        test_pred += m.predict_proba(X_test[cols])[:, 1] / N_FOLDS
    return oof, test_pred


PR, PRUNE_BEST, PRUNE_ACCEPTED = {}, None, False
if RUN_PRUNE:
    for K in PRUNE_K_GRID:
        t0 = time.time()
        for s in PRUNE_SCREEN:
            PR[(K, s)] = pruned_cv(K, s)
        ks_k = {s: ks_score(y, PR[(K, s)][0]) for s in PRUNE_SCREEN}
        print(f'  K={K}: KS per seed {({s: round(v, 4) for s, v in ks_k.items()})} | {(time.time() - t0) / 60:.1f} menit')
    screen = pd.DataFrame.from_dict({K: compare({s: ks_score(y, PR[(K, s)][0]) for s in PRUNE_SCREEN}, PRUNE_SCREEN) for K in PRUNE_K_GRID}, orient='index')
    display(screen[['delta', 'per_seed']].round(4))
    best_K = screen['delta'].astype(float).idxmax()
    if screen.loc[best_K, 'delta'] > 0:
        for s in PRUNE_CONFIRM:
            PR[(best_K, s)] = pruned_cv(best_K, s)
        conf = compare({s: ks_score(y, PR[(best_K, s)][0]) for s in PRUNE_CONFIRM}, PRUNE_CONFIRM)
        print(f'Konfirmasi K={best_K} (seed {PRUNE_CONFIRM}): {({k: (round(v, 4) if isinstance(v, float) else v) for k, v in conf.items()})}')
        PRUNE_BEST, PRUNE_ACCEPTED = best_K, conf['diterima']
    else:
        print('Tidak ada K yang melampaui baseline pada penyaringan; konfirmasi dilewati')
    print('Pemangkasan fitur:', f'DITERIMA (K={PRUNE_BEST})' if PRUNE_ACCEPTED else 'tidak diterima')

### 15.1 Bagging model terpangkas (hanya jika diterima)
Jika pemangkasan lolos konfirmasi, model terpangkas dilatih pada semua seed yang punya model dasar, lalu OOF dan prediksi test dirata-ratakan, sehingga dapat dibandingkan dengan bagging biasa.

In [ ]:
oof_pr, test_pr = None, None
if RUN_PRUNE and PRUNE_ACCEPTED:
    seeds_pr = sorted(all_models)
    for s in seeds_pr:
        if (PRUNE_BEST, s) not in PR:
            PR[(PRUNE_BEST, s)] = pruned_cv(PRUNE_BEST, s)
    oof_pr = np.mean([PR[(PRUNE_BEST, s)][0] for s in seeds_pr], axis=0)
    test_pr = np.mean([PR[(PRUNE_BEST, s)][1] for s in seeds_pr], axis=0)
    print(f'Bagging terpangkas (K={PRUNE_BEST}, {len(seeds_pr)} seed): KS OOF = {ks_score(y, oof_pr):.4f} | LightGBM {len(oofs10)} seed = {ks_score(y, oof_bag10):.4f}')
else:
    print('Model terpangkas tidak dipakai (tidak diterima atau dilewati)')

## 16. Langkah 3: kurva belajar dan refit seluruh data
**Kurva belajar.** Untuk tiap fold, model dilatih hanya pada 80% dari data latih fold (jadi 64% dari seluruh data) dan dinilai pada fold validasi yang sama. Selisih KS dibanding model 80% (baseline) menunjukkan seberapa besar data tambahan membantu. Jika kenaikannya kecil atau tidak konsisten, memakai 100% data tidak akan memberi apa-apa.

**Refit.** Model dilatih pada seluruh data train dengan jumlah pohon = rata-rata iterasi terbaik antar fold x 1.1 (data 25% lebih banyak membutuhkan sedikit lebih banyak pohon). Refit tidak dapat divalidasi langsung karena tidak ada data held-out, sehingga bukti satu-satunya adalah kurva belajar.

In [ ]:
from sklearn.model_selection import train_test_split


def small_cv(seed, frac=LC_FRAC):
    oof = np.zeros(len(X))
    for k, (i_tr, i_va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(X, y)):
        i_sub, _ = train_test_split(i_tr, train_size=frac, stratify=y.iloc[i_tr], random_state=seed + k)
        m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed + k)
        m.fit(X.iloc[i_sub], y.iloc[i_sub], eval_set=[(X.iloc[i_va], y.iloc[i_va])], eval_metric='auc',
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
        oof[i_va] = m.predict_proba(X.iloc[i_va])[:, 1]
    return oof


REFIT_OK, test_refit = False, None
if RUN_REFIT:
    t0 = time.time()
    lc = {s: {'KS_64%': ks_score(y, small_cv(s)), 'KS_80%': per_seed_ks[s]} for s in LC_SEEDS}
    lc_tab = pd.DataFrame(lc).T
    lc_tab['kenaikan_data_80%'] = lc_tab['KS_80%'] - lc_tab['KS_64%']
    display(lc_tab.round(4))
    gain_lc = lc_tab['kenaikan_data_80%']
    REFIT_OK = bool(gain_lc.mean() > ACCEPT_MIN and (gain_lc > 0).all())
    print(f'Kurva belajar: kenaikan KS dari 64% ke 80% data = {gain_lc.mean():+.4f} ({(time.time() - t0) / 60:.1f} menit)')
    print('Refit seluruh data didukung kurva belajar' if REFIT_OK else 'Kurva belajar tidak mendukung refit (kenaikan kecil atau tidak konsisten pada kedua seed)')

    preds = []
    for s in REFIT_SEEDS:
        n_iter = int(np.mean([m.best_iteration_ for m in all_models[s]]) * REFIT_ITER_FACTOR)
        m = lgb.LGBMClassifier(**{**LGB_PARAMS, 'n_estimators': n_iter}, random_state=s).fit(X, y)
        preds.append(m.predict_proba(X_test)[:, 1])
        print(f'  refit seed {s}: {n_iter} pohon')
    test_refit = np.mean(preds, axis=0)
    print(f"Korelasi peringkat prediksi test refit vs model fold (bag 5 seed): {pd.Series(test_bag).corr(pd.Series(test_refit), method='spearman'):.4f}")

## 17. Perbandingan resep dengan bobot tetap
Semua resep memakai bobot yang sama dengan blend saat ini (0.7 LightGBM : 0.3 regresi logistik); yang berubah hanya komponen LightGBM-nya. `R0` adalah blend yang sudah disubmit (skor Kaggle 0.55173). Setiap resep dibandingkan dengan `R0` memakai bootstrap berpasangan 200 kali atas OOF: selisih KS, interval kepercayaan 95%, dan peluang selisih > 0. Resep `R2` (refit) memakai OOF yang sama dengan `R1` karena refit tidak punya OOF sendiri.

In [ ]:
rank01 = lambda a: pd.Series(np.asarray(a)).rank(pct=True).to_numpy()
y_arr = y.to_numpy()

RECIPES = {   # kunci: (deskripsi, OOF LightGBM, prediksi test LightGBM)
    'R0': ('LightGBM 5 seed + logistik (sudah disubmit)', oof_bag, test_bag),
    'R1': (f'LightGBM {len(oofs10)} seed + logistik', oof_bag10, test_bag10),
}
if test_refit is not None:
    RECIPES['R2'] = ('LightGBM refit seluruh data + logistik (OOF = R1)', oof_bag10, test_refit)
if oof_pr is not None:
    RECIPES['R3'] = (f'LightGBM terpangkas K={PRUNE_BEST} + logistik', oof_pr, test_pr)

W_LR = 0.3
mix = lambda lg, lr_: (1 - W_LR) * rank01(lg) + W_LR * rank01(lr_)

rng, n = np.random.default_rng(0), len(y_arr)
boot_idx = [rng.integers(0, n, n) for _ in range(200)]
ks_boot = lambda p: np.array([ks_score(y_arr[i], p[i]) for i in boot_idx])

p0 = mix(oof_bag, oof_lr)
kb0, rows = ks_boot(p0), {}
for key, (desc, o_lg, t_lg) in RECIPES.items():
    p = mix(o_lg, oof_lr)
    d = ks_boot(p) - kb0
    rows[key] = {'resep': desc, 'KS_OOF': ks_score(y_arr, p), 'selisih_vs_R0': ks_score(y_arr, p) - ks_score(y_arr, p0),
                 'ci_bawah': np.percentile(d, 2.5), 'ci_atas': np.percentile(d, 97.5), 'peluang_>0': (d > 0).mean(),
                 'terbukti': bool(np.percentile(d, 2.5) > 0) if key != 'R0' else '-'}
recipe_tab = pd.DataFrame(rows).T
display(recipe_tab.round(4))

## 18. File submission kandidat
Untuk setiap resep selain `R0` dibuat file submission (divalidasi oleh `make_submission`), lengkap dengan korelasi peringkat prediksi test terhadap `R0`. Karena semua resep berbagi komponen yang sama, korelasinya akan sangat tinggi (di atas 0.99), sehingga skor leaderboard hampir tidak bisa membedakannya; gunakan bukti CV di bagian 17 untuk memilih.

In [ ]:
test_r0 = mix(test_bag, test_lr)
cand_files = {}
for key, (desc, o_lg, t_lg) in RECIPES.items():
    if key == 'R0':
        continue
    ks_k = float(recipe_tab.loc[key, 'KS_OOF'])
    path = f'{SUB_DIR}/sub_power_{key}_ks{ks_k:.4f}.csv'
    make_submission(X_test.index, mix(t_lg, test_lr), SUB, path)
    cand_files[key] = path
    rho_k = pd.Series(mix(t_lg, test_lr)).corr(pd.Series(test_r0), method='spearman')
    print(f'{key}: {desc}')
    print(f'    {path} | korelasi peringkat dengan R0: {rho_k:.4f}')

## 19. Ringkasan keputusan

In [ ]:
print('RINGKASAN TAHAP PENINGKATAN AKHIR')
print(f'  R0 (sudah disubmit): KS OOF = {ks_score(y_arr, p0):.4f}')
for key in recipe_tab.index.drop('R0'):
    r = recipe_tab.loc[key]
    print(f"  {key}: KS OOF = {float(r['KS_OOF']):.4f} | selisih vs R0 = {float(r['selisih_vs_R0']):+.4f} | interval 95% [{float(r['ci_bawah']):+.4f}, {float(r['ci_atas']):+.4f}] | terbukti = {r['terbukti']}")
print(f'  Pemangkasan fitur diterima: {PRUNE_ACCEPTED}' + (f' (K={PRUNE_BEST})' if PRUNE_ACCEPTED else ''))
print(f'  Kurva belajar mendukung refit: {REFIT_OK}')
print()
print('Panduan: R1 hanya menambah seed (pengurangan varians, tanpa pemilihan model) sehingga aman dipakai meski kenaikannya kecil.')
print('R2 layak dipakai hanya jika kurva belajar mendukung. R3 hanya jika pemangkasan diterima dan terbukti pada bootstrap.')